# Debug MILP – turing2_C5_V10

Instancias con `feasible=True` pero `valid=False` en el experimento `turing2_C5_V10__seed42__git81b49ed`.
Casos fallidos observados en la DB: **i1, i3, i8**.


In [1]:
import sys
sys.path.insert(0, '..')

## Configuración del experimento (igual a turing2_C5_V10)

In [2]:
cfg_experiment = {
    'num_customers': 5,
    'num_vehicles': 10,
    'capacity': [100],
    'vehicle_penalty': 0,
    'customer_demand': 1,
    'seed': 42,
}

cfg_generator = {
    'cost_range': 'low',
    'matrix_costs_ranges': {
        'low': [1, 20],
        'high': [100, 200],
        'high_dispersion': [1, 300],
    },
    'tw_tightness': 'loose',
    'time_windows_tightness': {
        'loose': [0, 500],
        'tight': [0, 20],
        'critical': [0, 7],
    },
    'time_windows_values': [5, 10],
}

NUM_INSTANCES = 10
SOLVERS_TO_TEST = ['cbc', 'glpk', 'highs', 'gurobi']  # los que fallaron en el experimento

> **Nota sobre capacity:** el yaml base usa `capacity: [100]`, pero fijate qué valor usó el experimento turing2.
> Si los resultados no coinciden con la DB, ajustá el valor acá.

## Generar las 10 instancias

El generador usa un único RNG inicializado con `seed=42` y lo consume secuencialmente.
Para reproducir exactamente las mismas instancias hay que llamar `generate()` 10 veces en orden.

In [3]:
from data.generators.synthetic import SyntheticGenerator

gen = SyntheticGenerator(cfg_experiment=cfg_experiment, cfg_generator=cfg_generator)

instances = []
for i in range(NUM_INSTANCES):
    print(f"\n{'='*50}")
    print(f"Generando instancia i{i}")
    print(f"{'='*50}")
    inst = gen.generate()
    instances.append(inst)
    print(inst)


Generando instancia i0
[C1] OK — demanda total=5, capacidad flota=1000.
[C2] OK — demanda por cliente={1}, Q_max=100.
[C3] OK — todas las ventanas tienen ei < li.
[C4] OK — todos li <= T_MAX=67.
[C5] OK — todos los clientes son alcanzables desde al menos un depot.
[C6] OK — todos los clientes pueden retornar a algún nodo de retorno.
[C7] Pares cliente-cliente alcanzables en tiempo: 20/20 (100.0%).
[C8] OK — vehículos disponibles=10, mínimo necesario=1.

  ✓  Instancia 'synthetic_n5_v10_seed42': pasa todas las verificaciones de factibilidad.

ProblemInstance: synthetic_n5_v10_seed42
----------------------------------------
Vehicles: 10
Customers: 5
Total nodes: 25

Customers: [0, 1, 2, 3, 4]
Depots: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Returns: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

Vehicle capacities (Q): [100, 100, 100, 100, 100, 100, 100, 100, 100, 100]
Fixed costs (F): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Customer time windows (ei, li):
  Customer 0: [0, 67]
  Customer 1: [2, 44]
  Cu

## Correr MILP en cada instancia y validar

In [4]:
from solvers.milp2a1 import MILPSolver
from validators.solution_validator import SolutionChecker
from time import time

results = []  # lista de dicts con todos los resultados

for solver_name in SOLVERS_TO_TEST:
    solver = MILPSolver(cfg_milp={'solver': solver_name})
    print(f"\n{'#'*60}")
    print(f"  Solver: {solver_name}")
    print(f"{'#'*60}")

    for i, inst in enumerate(instances):
        print(f"\n--- i{i} ({solver_name}) ---")
        try:
            t0 = time()
            solution = solver.solve(inst)
            runtime = time() - t0

            valid = False
            if solution.feasible:
                checker = SolutionChecker(insta=inst)
                valid = checker.check(solution.routes, solver_name)

            status = 'ok'
        except Exception as e:
            solution = None
            runtime = time() - t0
            valid = False
            status = f'error: {e}'

        row = {
            'instance': f'i{i}',
            'solver': solver_name,
            'feasible': solution.feasible if solution else None,
            'valid': valid,
            'cost': solution.total_cost if solution else None,
            'runtime': round(runtime, 4),
            'routes': solution.routes if solution else None,
            'unserved': solution.unserved_customers if solution else None,
            'status': status,
        }
        results.append(row)
        print(f"  feasible={row['feasible']}  valid={row['valid']}  cost={row['cost']}  runtime={row['runtime']}s")
        if not valid and solution and solution.feasible:
            print(f"  *** CASO PROBLEMATICO: feasible=True pero valid=False ***")
            print(f"  routes: {solution.routes}")


############################################################
  Solver: cbc
############################################################

--- i0 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True  valid=True  cost=38.0  runtime=4.9666s

--- i1 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True  valid=True  cost=38.0  runtime=28.7896s

--- i2 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True  valid=True  cost=37.0  runtime=4.3366s

--- i3 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True  valid=True  cost=34.0  runtime=5.9095s

--- i4 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True  valid=True  cost=29.0  runtime=4.6538s

--- i5 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True  valid=True  cost=31.0  runtime=7.885s

--- i6 (cbc) ---
INFO [SolutionChecker]: valid solution for solver: cbc
  feasible=True 

## Resumen

In [5]:
import pandas as pd

df = pd.DataFrame(results)
df

,instance,solver,feasible,valid,cost,runtime,routes,unserved,status
0,i0,cbc,True,True,38.0,4.9666,"{7: [7, 2, 3, 0, 17], 13: [13, 1, 4, 23]}",[],ok
1,i1,cbc,True,True,38.0,28.7896,"{7: [7, 1, 2, 0, 4, 21, 3, 17]}",[],ok
2,i2,cbc,True,True,37.0,4.3366,"{7: [7, 2, 13, 1, 0, 17], 8: [8, 3, 4, 18]}",[],ok
3,i3,cbc,True,True,34.0,5.9095,"{8: [8, 2, 12, 10, 3, 1, 18], 13: [13, 0, 4, 23]}",[],ok
4,i4,cbc,True,True,29.0,4.6538,"{7: [7, 2, 13, 1, 16, 3, 14, 0, 4, 17]}",[],ok
5,i5,cbc,True,True,31.0,7.8850,"{9: [9, 4, 1, 19], 10: [10, 3, 0, 2, 20]}",[],ok
6,i6,cbc,True,True,29.0,3.6148,"{10: [10, 4, 18, 0, 20], 13: [13, 2, 12, 1, 3,...",[],ok
7,i7,cbc,True,True,31.0,5.2452,"{9: [9, 2, 3, 12, 0, 4, 1, 19]}",[],ok
8,i8,cbc,True,True,32.0,6.9042,"{6: [6, 1, 4, 16], 12: [12, 3, 20, 0, 23, 2, 1...",[],ok
9,i9,cbc,True,True,37.0,5.9323,"{8: [8, 4, 1, 2, 3, 20, 14, 0, 10, 18]}",[],ok


In [6]:
# Solo los casos problemáticos
df[(df['feasible'] == True) & (df['valid'] == False)]

,instance,solver,feasible,valid,cost,runtime,routes,unserved,status


## Inspección profunda de las instancias problemáticas

Tomamos cada instancia que produjo `feasible=True, valid=False` y la inspeccionamos manualmente.

In [7]:
failing = df[(df['feasible'] == True) & (df['valid'] == False)]

for _, row in failing.drop_duplicates(subset='instance').iterrows():
    idx = int(row['instance'][1:])
    inst = instances[idx]

    print(f"\n{'='*60}")
    print(f"Instancia {row['instance']}")
    print(f"{'='*60}")
    print(inst)

    print("\nTW completo (todos los nodos):")
    for n in range(inst.N):
        tag = 'C' if n in inst.C else ('D' if n in inst.DEPOTS else 'R')
        print(f"  nodo {n:2d} [{tag}]: ei={inst.ei[n]:5d}  li={inst.li[n]:5d}")

    print(f"\nRutas:")
    for solver_name in SOLVERS_TO_TEST:
        r = df[(df['instance'] == row['instance']) & (df['solver'] == solver_name)].iloc[0]
        print(f"  {solver_name}: {r['routes']}")

    # Reconstrucción manual de la ruta para ver dónde falla la validación
    print("\nSimulación manual de la ruta (primer solver con fallo):")
    for solver_name in SOLVERS_TO_TEST:
        r = df[(df['instance'] == row['instance']) & (df['solver'] == solver_name)].iloc[0]
        if r['feasible'] and not r['valid'] and r['routes']:
            for depot_key, route in r['routes'].items():
                print(f"  Vehículo {depot_key}: {route}")
                t = inst.ei[route[0]]
                for j_idx in range(len(route) - 1):
                    i_node = route[j_idx]
                    j_node = route[j_idx + 1]
                    travel = inst.tij[i_node][j_node]
                    arrival = t + travel
                    tag_j = 'C' if j_node in inst.C else ('D' if j_node in inst.DEPOTS else 'R')
                    ok = arrival <= inst.li[j_node]
                    print(f"    {i_node} → {j_node}[{tag_j}]: t={t}  travel={travel}  arrival={arrival}  [ei={inst.ei[j_node]}, li={inst.li[j_node]}]  {'OK' if ok else '*** VIOLA TW ***'}")
                    t = max(arrival, inst.ei[j_node])
            break

conclusion: no se detecto nigun problema, y como fue para una sola instancia aislada me suena que pudo haber sido algun tipo de interupcion en la ejecucion.